# Lesson 01 - Bevezetés az AI ügynökökbe

Üdvözlünk az **AI Agents for Beginners** tanfolyam első leckéjében!

Egy **AI ügynök** olyan program, amely egy nagy nyelvi modellt (LLM) használ érvelési motorjaként, és képes *műveleteket* végrehajtani a való világban — API-k hívásával, adatbázisok lekérdezésével vagy kód futtatásával — hogy egy felhasználó nevében célt érjen el.

Ebben a jegyzetfüzetben elkészíted az első ügynöködet: egy **Utazási Ügynököt**, amely nyaralási célállomásokat ajánl. Eközben megtanulod, hogyan:

1. Csatlakozz az Azure AI Foundry Agent Service-hez a **Microsoft Agent Framework** segítségével.
2. Adj az ügynöknek egy **eszközt** — egy egyszerű Python függvényt, amit meghívhat.
3. Futtasd az ügynököt és vizsgáld meg a válaszát.
4. Tokenenként streameld az ügynök válaszát.


## Beállítás

A jegyzetfüzet futtatása előtt győződj meg arról, hogy:

1. **Van egy Azure AI Foundry projekted** telepített csevegőmodellel (pl. `gpt-4o-mini`).
2. **Be vagy jelentkezve az Azure CLI-vel** — futtasd az `az login` parancsot a terminálodban.
3. **Beállítottad a szükséges környezeti változókat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — az Azure AI Foundry projekted végpontja.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.

Az alábbi cella telepíti a szükséges Python csomagokat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Az első ügynök létrehozása

Egy ügynöknek két dologra van szüksége:

- **Utasítások**, amelyek megmondják neki, *ki ő* és *hogyan viselkedjen* (egy rendszer prompt).
- **Eszközök** — Python függvények, amelyeket a `@tool` dekorátorral jelölnek, és amelyeket az ügynök hívhat meg információk lekérésére vagy műveletek végrehajtására.

Alább definiálunk egy egyszerű eszközt, amely népszerű nyaralási célpontok listáját adja vissza. Az ügynök ezt az eszközt fogja használni, amikor egy felhasználó utazási ajánlásokat kér.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Válaszok

A még interaktívabb élmény érdekében az ügynök válaszát **folyamként** is megkaphatod. Ahelyett, hogy a teljes választ megvárnád, az ügynök a generálás során szövegrészeket ad át. Ez különösen hasznos chat felületeken, ahol valós időben szeretnéd megjeleníteni a kimenetet.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

- **Készíts szolgáltatót**, amely az Azure AI Foundry Agent Service-hez csatlakozik az `AzureAIProjectAgentProvider` segítségével.
- **Határozz meg egy eszközt** az `@tool` dekorátor segítségével, hogy az ügynök meghívhassa a Python függvényeidet.
- **Futtasd az ügynököt** egy felhasználói üzenettel, és írd ki a válaszát.
- **Sugározd a válaszokat** valós idejű kimenethez.

A következő leckében mélyebben megvizsgáljuk az ügynöki keretrendszereket, és megtanuljuk, hogyan adhatsz az ügynököknek hatékonyabb eszközöket és többlépcsős érvelési képességeket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Felmentés**:
Ezt a dokumentumot az AI fordítószolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk le. Bár igyekszünk pontosak lenni, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum anyanyelven készült változata tekintendő hiteles forrásnak. Fontos információk esetén szakmai emberi fordítást javasolunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
